# Kiểm thử Môi trường Phân tích & Mô phỏng Quy trình IE203 - Bệnh viện Vinmec (Task T30)

**Đồ án:** Phân tích, mô hình hóa và cải tiến quy trình khám chữa bệnh ngoại trú tại Bệnh viện Đa khoa Quốc tế Vinmec Central Park (IE203.G12).  
**Người thực hiện:** Đinh Hoàng Phát (MSSV: 26410086).  

Notebook này kiểm tra toàn bộ môi trường phục vụ đồ án:
1. Phiên bản Python và các thư viện cốt lõi (`pm4py`, `simpy`, `pandas`, `numpy`, `matplotlib`, `requests`).
2. Chạy mô phỏng hướng sự kiện rời rạc với **SimPy**.
3. Đọc `data/event_log.csv` (hoặc `data/event_log_template.csv`) với **Pandas** và khám phá DFG bằng **PM4Py**.
4. Kiểm tra đọc mô hình **BPMN 2.0** bằng **PM4Py** (`pm4py.read_bpmn`).
5. Trực quan hóa biểu đồ thời gian chu trình bằng **Matplotlib**.
6. Kiểm tra kết nối **Camunda BPM** local (Docker Compose tại `infra/docker-compose.yml`).

## 1. Kiểm tra Phiên bản Python và Thư viện Cốt lõi

In [ ]:
import platform
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import simpy
import pm4py
import requests

print(f"Python Version: {platform.python_version()}")
print(f"Pandas: {pd.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"SimPy: {simpy.__version__}")
print(f"PM4Py: {pm4py.__version__}")
print(f"Requests: {requests.__version__}")
print("\n-> Tất cả các thư viện lõi đã được cài đặt thành công!")

## 2. Kiểm thử Mô phỏng Hướng sự kiện Rời rạc (SimPy)

Mô phỏng một kịch bản đơn giản người bệnh ngoại trú đến quầy Tiếp đón và đo Sinh hiệu tại Vinmec Central Park.

In [ ]:
import random

random.seed(42)

def patient_journey(env, patient_id, nurse_resource, logs):
    arrival_time = env.now
    with nurse_resource.request() as req:
        yield req
        wait_time = env.now - arrival_time
        service_duration = round(random.uniform(3, 7), 2)
        yield env.timeout(service_duration)
        logs.append({
            'patient_id': patient_id,
            'wait_time_min': round(wait_time, 2),
            'service_time_min': service_duration,
            'end_time_min': round(env.now, 2)
        })

env = simpy.Environment()
nurse = simpy.Resource(env, capacity=2)
simulation_logs = []

for i in range(1, 6):
    env.process(patient_journey(env, f'L0{i}', nurse, simulation_logs))

env.run(until=60)
print(f"Mô phỏng 60 phút hoàn tất. Số ca hoàn thành: {len(simulation_logs)}")
pd.DataFrame(simulation_logs)

## 3. Kiểm thử Đọc & Xử lý Event Log (Pandas & PM4Py)

Đọc dữ liệu từ `../data/event_log.csv` (nếu đã có) hoặc `../data/event_log_template.csv`, sau đó chuyển đổi sang Event Log của PM4Py để khám phá đồ thị DFG.

In [ ]:
from pathlib import Path

log_path = Path("../data/event_log.csv")
if not log_path.exists():
    log_path = Path("../data/event_log_template.csv")

df_log = pd.read_csv(log_path)
df_log["start_time"] = pd.to_datetime(df_log["start_time"])
df_log["end_time"] = pd.to_datetime(df_log["end_time"])

print(f"Đã đọc {len(df_log)} sự kiện từ: {log_path}")
display(df_log.head())

event_log = pm4py.format_dataframe(
    df_log,
    case_id="case_id",
    activity_key="activity",
    timestamp_key="end_time"
)
dfg, start_activities, end_activities = pm4py.discover_dfg(event_log)
print("\nDirectly Follows Graph (DFG):")
for (act1, act2), count in dfg.items():
    print(f"  {act1} ---> {act2} : {count} lượt")

## 4. Kiểm thử Đọc Mô hình BPMN 2.0 (PM4Py)

In [ ]:
import tempfile

bpmn_files = sorted(Path("../bpmn").rglob("*.bpmn"))
if bpmn_files:
    bpmn_graph = pm4py.read_bpmn(str(bpmn_files[0]))
    print(f"Đã đọc file BPMN: {bpmn_files[0]} ({len(bpmn_graph.get_nodes())} nodes)")
else:
    sample_bpmn = """<?xml version="1.0" encoding="UTF-8"?>
<bpmn:definitions xmlns:bpmn="http://www.omg.org/spec/BPMN/20100524/MODEL"
                  id="Definitions_Test" targetNamespace="http://bpmn.io/schema/bpmn">
  <bpmn:process id="Process_Test" isExecutable="false">
    <bpmn:startEvent id="Start_1" name="Bắt đầu"/>
    <bpmn:task id="Task_1" name="Tiếp đón"/>
    <bpmn:endEvent id="End_1" name="Kết thúc"/>
    <bpmn:sequenceFlow id="Flow_1" sourceRef="Start_1" targetRef="Task_1"/>
    <bpmn:sequenceFlow id="Flow_2" sourceRef="Task_1" targetRef="End_1"/>
  </bpmn:process>
</bpmn:definitions>
"""
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir) / "sample.bpmn"
        tmp_path.write_text(sample_bpmn, encoding="utf-8")
        bpmn_graph = pm4py.read_bpmn(str(tmp_path))
        print(f"Đọc thành công BPMN mẫu bằng PM4Py: {len(bpmn_graph.get_nodes())} nodes")

## 5. Kiểm thử Trực quan hóa Biểu đồ Thời gian Chu trình (Cycle Time)

In [ ]:
case_durations = df_log.groupby("case_id").agg(start=("start_time", "min"), end=("end_time", "max"))
case_durations["duration_min"] = (case_durations["end"] - case_durations["start"]).dt.total_seconds() / 60

plt.figure(figsize=(7, 3.5))
plt.bar(case_durations.index, case_durations["duration_min"], color="#005baa", width=0.4)
plt.title("Tổng Thời gian Chu trình Khám Ngoại trú theo Ca (Phút) - Vinmec Central Park")
plt.xlabel("Mã ca bệnh (Case ID)")
plt.ylabel("Thời gian (Phút)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 6. Kiểm thử Kết nối Camunda BPM Local (Docker Compose)

Trước khi chạy cell này, hãy khởi động Camunda tại thư mục `infra/` bằng lệnh `docker compose up -d`.

In [ ]:
try:
    resp = requests.get("http://localhost:8080/engine-rest/version", timeout=2)
    print("Camunda REST API đang hoạt động:", resp.json())
except Exception as exc:
    print("Chưa kết nối được Camunda tại http://localhost:8080 (chạy `docker compose up -d` trong thư mục infra/).")
    print("Chi tiết:", exc)